# Chapter 10: Tool Engineering
## Topic 6: Multi-Tool Agents — `lookup_product_catalog` and `check_sender_history`

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Every topic in this chapter so far has focused on getting *one* tool right — `validate_fd_reference`'s design (Topic 3), its schema (Topic 4), its error handling (Topic 5). This topic asks what changes once an agent has *several* tools available at once, each individually well-designed, but now needing to coexist, be chosen between correctly, and potentially be called together in service of a single response.
- The core new problem multi-tool agents introduce, beyond anything a single-tool agent faces: **tool selection** — given several available tools, the model must correctly identify *which* tool (or which combination of tools) actually addresses the current request, not just *whether* to call a tool at all (Chapter 9 Topic 1's single-tool triggering question). This is a strictly harder problem than single-tool triggering, since it requires the model to correctly discriminate between multiple plausible options, not just decide yes-or-no against one.
- This project's two new example tools illustrate two genuinely different tool *categories*, worth distinguishing precisely: **`lookup_product_catalog`** is conceptually similar to `validate_fd_reference` — a lookup against a defined, bounded dataset (available FD products and their terms), directly connecting back to Chapter 9 Topic 8's Smart Saver FD proof (a new product added to a catalog like this is exactly the kind of content that tool would need to look up). **`check_sender_history`** is a different kind of tool — it doesn't look up a specific record the customer explicitly referenced, but instead queries *context about the customer themselves* (are they a repeat sender, what have they emailed about before) that the customer's current email doesn't explicitly ask for, but which could genuinely change how their current request should be handled.


### 2. Internal Working — Step by Step

**What genuinely changes, mechanically, once there are multiple tools:**

1. **The `TOOLS` list simply grows** — mechanically, this is a small change (Topic 2's message mechanics don't change at all; the model still receives the same kind of `tools` parameter, just with more entries). The real complexity is not mechanical, it's about tool *design coherence* — every tool in the list needs a name and description specific and distinct enough (Topic 4's principles) that the model can reliably discriminate between them, not just describe itself well in isolation.
2. **A single model turn can now request more than one tool call at once** — this is precisely where Topic 2's `tool_use_id` linkage mechanism, previously discussed but not yet load-bearing in this project's single-tool examples, becomes genuinely necessary: the response's `content` list can contain multiple `tool_use` blocks, each needing its own `id`, and the dispatch layer needs to process each one, calling potentially several different real functions, and package multiple `tool_result` blocks (each correctly tagged with its corresponding `tool_use_id`) into the next message.
3. **Tool *composition* becomes a real, new question the single-tool case never raised:** should `check_sender_history` inform whether `validate_fd_reference` or `lookup_product_catalog` even needs to be called at all, or should all relevant tools generally be called and their results combined afterward? This connects directly to Chapter 9 Topic 5's RAG chain patterns — the same Stuff/sequential/parallel-and-combine tension that topic named for combining retrieved chunks applies here too, just for combining multiple tools' results into one coherent final answer.
4. **The system prompt's role grows in importance, not just the individual tool descriptions** — with several tools available, overall guidance about *how* they relate to each other (for example, "always check sender history first, since a known repeat customer with unresolved prior issues may need different handling than a first-time sender") becomes a genuinely new kind of instruction beyond what any single tool's own description could capture on its own.


### 3. How This Is Implemented in This Project

- **`lookup_product_catalog`** follows exactly the same design template Topic 3 established for `validate_fd_reference`: a tool-facing function with a clear found/not-found (or, here, exists/doesn't-exist) distinction, structured multi-field output (product name, rate, tenure, minimum deposit — not a flattened description string), and a schema (Topic 4) with an explicit when-to-call and when-not-to-call boundary. This tool directly serves the Chapter 9 Topic 8 Smart Saver FD scenario: a customer asking about a specific named product is exactly the case this tool exists for.
- **`check_sender_history`** is architecturally different in one important way: rather than being triggered by an explicit reference in the customer's email (like a specific FD number or product name), it's more naturally called *proactively*, at the start of handling most or all emails, since knowing whether this is a repeat sender is broadly useful context regardless of what the email's specific content turns out to be — this connects to the project's own real, measured finding referenced in the broader syllabus (a documented repeat-sender rate in the corpus), which is precisely the kind of finding that motivates building a tool like this at all.
- The dispatch layer's `if/elif` chain (Topic 1, Topic 2) simply grows to include a new branch for each new tool name — `lookup_product_catalog` and `check_sender_history` each get their own case, calling their own real, separately-designed functions, following exactly the same real-function-behind-a-schema separation principle established from Topic 1 onward.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Tool selection ambiguity grows with the number of tools, not linearly but in terms of genuine confusability** — two tools is a fairly easy discrimination problem; five or six tools with even slightly overlapping scopes becomes a meaningfully harder problem for the model to get right consistently, directly connecting to Topic 4's schema-clarity discussion, now under greater pressure as the tool list grows.
- **Multiple tool calls in a single turn compound both cost and latency** — Topic 2 already established the compounding cost of a growing message list across turns; calling several tools within a single turn adds this same effect *within* one turn, since every one of those tool calls' results gets added to the message history before the model's next generation call, which then resends everything.
- **A proactively-called tool like `check_sender_history` raises a genuine design question about when it should run** — calling it on every single email regardless of content adds a small, real cost to every request; calling it only conditionally (for example, only once an email is otherwise ambiguous, per Chapter 9 Topic 1's triggering discussion) saves cost but might miss context that would have been useful for a case that didn't otherwise look ambiguous. This is a real trade-off, not an obviously correct default.
- **Debugging a wrong-tool-selected case requires checking whether it's a scope-overlap problem (Topic 4's schema clarity) or a genuine multi-tool composition problem** (the model correctly identified relevant tools individually, but combined or sequenced their results incorrectly) — these have different fixes, and conflating them wastes debugging effort exactly as earlier topics warned about conflating different failure categories.
- **Monitoring:** for a multi-tool agent, tracking which *combinations* of tools get called together for a given request type (not just each tool's individual call rate) reveals whether the model's tool-composition behavior is consistent and sensible — a repeat-sender question that sometimes triggers `check_sender_history` and sometimes doesn't, for very similar-looking emails, is a specific, trackable inconsistency worth investigating.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Proactive (always-called) tools vs reactive (triggered-by-specific-content) tools:** `validate_fd_reference` and `lookup_product_catalog` are naturally reactive — called when the email explicitly references something specific they can look up. `check_sender_history` is naturally more proactive — broadly useful context regardless of specific content. Deciding which category a new tool belongs to should follow from what the tool's information is actually *for*, not be assumed uniformly across every tool in the agent.
- **Should tool composition logic live in the system prompt, or be left entirely to the model's own judgment given the individual tool descriptions?** For a genuinely important sequencing or composition rule (like "always check sender history before finalizing, if this looks like a complaint"), making it an explicit system-prompt instruction is more reliable than hoping the model infers the right composition purely from reading several independent tool descriptions — this mirrors exactly Chapter 9 Topic 3's finding that mandatory tool-use rules belong in the system prompt, not solely in a tool's own description.
- **How many tools is "too many" for one agent to reliably manage?** There's no universal fixed number, but the real signal is measured tool-selection accuracy (this chapter's upcoming testing topic) degrading as more tools are added — at that point, splitting into multiple more narrowly-scoped agents, or restructuring the tool set for clearer, less-overlapping scopes, becomes the right fix rather than continuing to add tools to an already-strained single agent.


### 6. Alternatives and When to Use Each

- **A single, general-purpose agent with all tools available (this project's current approach):** appropriate while the tool count stays small enough that tool-selection accuracy remains measurably high — the right starting point for most projects.
- **Splitting into multiple, more narrowly-scoped agents, each with a smaller tool set:** worth considering once measured tool-selection accuracy degrades as more tools are added to one agent, or once different tools serve genuinely different stages of a workflow that don't need to coexist in the same decision-making context.
- **A router/dispatcher pattern, where a first, lightweight step decides which specialized agent (each with its own smaller tool set) should handle a request:** a more complex architecture worth adopting once tool-selection complexity genuinely can't be managed within one agent's tool list — not necessary for this project's current, modest tool count.


### 7. Common Mistakes and Production Failures

- Adding new tools to an agent's tool list without checking for name or description overlap with existing tools (Topic 4's principles), increasing the chance of tool-selection confusion as the list grows.
- Treating a naturally proactive tool (like `check_sender_history`) as if it should only be reactively triggered by explicit content, missing context that would have been useful for cases that don't obviously call for it.
- Not accounting for the compounding cost of multiple tool calls within a single turn, on top of the already-compounding cost of a growing message list across turns.
- Leaving important tool-composition rules (which tools should be called together, in what order, under what conditions) entirely to the model's own inference from individual tool descriptions, rather than stating them explicitly in the system prompt when they're genuinely important.
- Continuing to add tools to a single, general-purpose agent indefinitely without measuring whether tool-selection accuracy is degrading, rather than treating agent architecture (single vs multiple specialized agents) as a decision that should respond to measured evidence.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What new problem does a multi-tool agent introduce that a single-tool agent doesn't have?
  A: Tool selection — the model must correctly discriminate between multiple available tools to identify which one (or which combination) actually addresses the current request, not just decide yes-or-no about whether to call a single available tool. This is a strictly harder problem, since it requires distinguishing between plausible alternatives, not just a binary triggering decision.

- Q: What's the architectural difference between `lookup_product_catalog` and `check_sender_history` as tool categories?
  A: `lookup_product_catalog` is naturally reactive — triggered by the customer's email explicitly referencing something specific (a named product) that it can look up, similar to `validate_fd_reference`. `check_sender_history` is naturally more proactive — it provides broadly useful context about the customer (are they a repeat sender) that isn't necessarily explicitly requested by the email's specific content, but that could still usefully inform how the request should be handled.

**Intermediate**

- Q: Why does `tool_use_id` linkage (introduced conceptually in Topic 2) become genuinely necessary, rather than just a good practice, once an agent has multiple tools?
  A: A single model turn can request more than one tool call at once in a multi-tool agent — the response's content list can contain several `tool_use` blocks. Without correctly matching each result back to its specific request via `tool_use_id`, there would be no reliable way for the model to know which of several results corresponds to which of its several requests, especially if, say, two different reference-number lookups were both requested in the same turn.

- Q: How would you decide whether `check_sender_history` should be called on every email, or only under specific conditions?
  A: This is a real cost-versus-completeness trade-off, not an obviously correct default. Calling it on every email guarantees the context is always available but adds a small cost to every single request; calling it conditionally (for example, only when an email otherwise looks ambiguous, per Chapter 9 Topic 1's triggering framework) saves cost but risks missing potentially useful context for cases that don't superficially look like they need it. The right choice should be informed by measuring how often that context actually changes the final response's quality or correctness when it's available versus when it isn't.

**Advanced**

- Q: An agent's tool list grows to include several tools with genuinely overlapping scope. How would you decide whether to fix this by rewriting the schemas (Topic 4) versus splitting into multiple specialized agents?
  A: Start by measuring tool-selection accuracy (this chapter's testing topic) to confirm the overlap is actually causing measurable confusion, not just a theoretical concern. If schema rewrites — sharpening names and descriptions, adding explicit when-not-to-call boundaries distinguishing the overlapping tools — measurably resolve the confusion on a labeled test set, that's the lower-complexity fix. If overlap persists even after genuinely careful schema work, or if the tools serve conceptually different workflow stages that don't naturally belong in the same decision context, splitting into multiple, more narrowly-scoped agents becomes the more appropriate, if more complex, architectural fix.

- Q: Design the composition logic for a case where `check_sender_history` reveals the customer is a repeat sender with a previously unresolved issue, and the current email is also asking a specific FD product question that would trigger `lookup_product_catalog`. How should these two tools' results be combined?
  A: Both tools would likely be called in the same turn or in close sequence — `check_sender_history` to surface the unresolved prior issue, `lookup_product_catalog` to answer the specific product question. The generation step then needs to synthesize both: answering the immediate product question using the catalog lookup's structured result, while also acknowledging or escalating the unresolved prior issue using the sender-history result — this composition logic (how to weave together results from genuinely different tool categories into one coherent response) is exactly the kind of guidance that belongs in the system prompt rather than being left to inference, mirroring Chapter 9 Topic 5's RAG chain-pattern discussion about combining multiple pieces of information into one coherent final answer.

**Scenario-based**

- Q: Production monitoring shows that for very similar-looking customer emails, the model sometimes calls `check_sender_history` and sometimes doesn't, with no clear pattern. Diagnose.
  A: This inconsistency suggests the system prompt (or the tool's own description) doesn't give clear enough guidance about when this genuinely proactive tool should be called — since it's not naturally triggered by explicit email content the way a reactive tool is, its calling condition needs to be stated more explicitly and consistently, likely as a system-prompt-level rule ("always check sender history for X category of email") rather than left to the model's own judgment about relevance case by case, which produces exactly this kind of inconsistent behavior.

**Follow-up questions to expect:**

- "How would you design a test to measure tool-selection accuracy specifically, as distinct from single-tool triggering accuracy?"
- "What would you monitor to detect that an agent has too many tools for reliable tool selection?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **Tool selection among many options is a specific instance of a general classification problem** — choosing the correct action from several discrete options based on context is a well-studied problem shape appearing throughout machine learning and decision-making systems generally, not unique to LLM tool-calling; recognizing this connects tool-selection reliability concerns to a much broader body of applicable technique and intuition (like Chapter 7's classification-adjacent retrieval-triggering work).
- **The proactive-vs-reactive tool distinction generalizes to a broader question in agent design: what context should always be gathered upfront versus fetched only on demand** — this shows up well beyond this specific project, in any system that needs to balance the cost of gathering potentially-unnecessary context against the risk of missing genuinely useful context that wasn't obviously needed in advance.
- **This topic is the direct, practical bridge to Chapter 12's agent orchestration (LangGraph) discussion** — as tool composition logic (which tools to call, in what order, under what conditions) grows more complex than a single system prompt can cleanly express, a more structured orchestration framework becomes the natural next step, exactly the motivation Chapter 12 will formalize.

### 10. Quick Revision Summary (for last-minute interview prep)

> Multi-tool agents introduce tool *selection* — correctly discriminating between multiple available tools, not just deciding whether to call one — as a genuinely harder problem than single-tool triggering. `lookup_product_catalog` follows the same reactive, found/not-found, structured-output design template as `validate_fd_reference` (Topic 3), directly serving the Chapter 9 Topic 8 Smart Saver FD scenario. `check_sender_history` is architecturally different: a naturally proactive tool providing broadly useful customer context that isn't necessarily triggered by specific email content, raising a real cost-versus-completeness trade-off about when it should actually be called. As the tool list grows, `tool_use_id` linkage (Topic 2) becomes genuinely load-bearing rather than a theoretical nicety, since a single turn can now request multiple tool calls at once. Tool composition — how multiple tools' results get combined into one coherent final answer — is a new question multi-tool agents raise that single-tool agents never face, and important composition rules should be stated explicitly in the system prompt rather than left entirely to the model's inference from individual tool descriptions. As tool count and scope overlap grow, measured tool-selection accuracy (not intuition) should determine whether the fix is sharper schemas (Topic 4) or splitting into multiple, more narrowly-scoped specialized agents.


### Module 1: Two New Tools, Same Design Template as `validate_fd_reference`

Build `lookup_product_catalog` and `check_sender_history`, each following Topic 3's design principles -- structured output, clear found/not-found (or exists/new) distinction.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Two new tools, following validate_fd_reference's template
# ------------------------------------------------------------------

PRODUCT_CATALOG = {
    "smart saver fd": {"interest_rate_percent": 7.65, "tenure_months": 18, "minimum_deposit_inr": 25000},
    "senior citizen fd": {"interest_rate_percent": 7.5, "tenure_months": 24, "minimum_deposit_inr": 10000},
}

def lookup_product_catalog(product_name: str) -> dict:
    """Same design template as validate_fd_reference: structured
    output, explicit exists/not-exists distinction, echoes input back."""
    key = product_name.strip().lower()
    product = PRODUCT_CATALOG.get(key)
    if product is None:
        return {"product_name": product_name, "exists": False}
    return {"product_name": product_name, "exists": True, **product}


SENDER_HISTORY = {
    "shobha.chopra@email.com": {"prior_email_count": 3, "has_unresolved_issue": True,
                                  "last_topic": "premature withdrawal penalty dispute"},
    "new.customer@email.com": {"prior_email_count": 0, "has_unresolved_issue": False, "last_topic": None},
}

def check_sender_history(sender_email: str) -> dict:
    """Same design template: structured output, echoes input, clear
    is_repeat_sender distinction (this tool's version of found/not-found)."""
    history = SENDER_HISTORY.get(sender_email.strip().lower())
    if history is None or history["prior_email_count"] == 0:
        return {"sender_email": sender_email, "is_repeat_sender": False, "prior_email_count": 0}
    return {"sender_email": sender_email, "is_repeat_sender": True, **history}


print("=" * 70)
print("MODULE 1: TWO NEW TOOLS, SAME DESIGN TEMPLATE")
print("=" * 70)

print("\nlookup_product_catalog examples:")
for product in ["Smart Saver FD", "Nonexistent Product"]:
    print(f"  '{product}' -> {lookup_product_catalog(product)}")

print("\ncheck_sender_history examples:")
for email in ["shobha.chopra@email.com", "brand.new.sender@email.com"]:
    print(f"  '{email}' -> {check_sender_history(email)}")

print("\nBoth tools follow the SAME design principles as validate_fd_reference:")
print("structured multi-field output, explicit found/exists distinction,")
print("original input echoed back in every result.")
print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: TWO NEW TOOLS, SAME DESIGN TEMPLATE

lookup_product_catalog examples:
  'Smart Saver FD' -> {'product_name': 'Smart Saver FD', 'exists': True, 'interest_rate_percent': 7.65, 'tenure_months': 18, 'minimum_deposit_inr': 25000}
  'Nonexistent Product' -> {'product_name': 'Nonexistent Product', 'exists': False}

check_sender_history examples:
  'shobha.chopra@email.com' -> {'sender_email': 'shobha.chopra@email.com', 'is_repeat_sender': True, 'prior_email_count': 3, 'has_unresolved_issue': True, 'last_topic': 'premature withdrawal penalty dispute'}
  'brand.new.sender@email.com' -> {'sender_email': 'brand.new.sender@email.com', 'is_repeat_sender': False, 'prior_email_count': 0}

Both tools follow the SAME design principles as validate_fd_reference:
structured multi-field output, explicit found/exists distinction,
original input echoed back in every result.

Module 1 complete. Run Module 2 next.


### Module 2: Real Multi-Tool Dispatch — Multiple `tool_use` Blocks in One Turn

Simulate a single model response requesting THREE different tool calls at once, and run the real dispatch logic handling all of them with correct tool_use_id linkage.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Real multi-tool dispatch, multiple tool_use blocks at once
# ------------------------------------------------------------------

import json

def validate_fd_reference(reference_number: str) -> dict:
    FD_RECORDS = {"BJ2019FD7717": {"customer_name": "Shobha Chopra", "status": "Closed (Premature)"}}
    record = FD_RECORDS.get(reference_number.strip())
    if record is None:
        return {"reference_number": reference_number, "found": False}
    return {"reference_number": reference_number, "found": True, **record}


def simulate_multi_tool_model_response() -> dict:
    """Simulates a SINGLE model turn requesting THREE tool calls at
    once -- this is exactly the scenario that makes tool_use_id
    linkage genuinely necessary, not just a nice-to-have."""
    return {
        "stop_reason": "tool_use",
        "content": [
            {"type": "tool_use", "id": "toolu_A", "name": "check_sender_history",
             "input": {"sender_email": "shobha.chopra@email.com"}},
            {"type": "tool_use", "id": "toolu_B", "name": "validate_fd_reference",
             "input": {"reference_number": "BJ2019FD7717"}},
            {"type": "tool_use", "id": "toolu_C", "name": "lookup_product_catalog",
             "input": {"product_name": "Smart Saver FD"}},
        ],
    }


def dispatch_multi_tool_call(response: dict) -> list:
    """REAL dispatch logic handling MULTIPLE tool_use blocks in one
    turn -- each result correctly tagged with its OWN tool_use_id,
    matching Topic 2's message mechanics, now genuinely load-bearing
    with three DIFFERENT tools in a single turn."""
    tool_result_blocks = []
    for block in response["content"]:
        if block["type"] != "tool_use":
            continue

        if block["name"] == "check_sender_history":
            result = check_sender_history(block["input"]["sender_email"])
        elif block["name"] == "validate_fd_reference":
            result = validate_fd_reference(block["input"]["reference_number"])
        elif block["name"] == "lookup_product_catalog":
            result = lookup_product_catalog(block["input"]["product_name"])
        else:
            result = {"error": "unknown_tool"}

        tool_result_blocks.append({
            "type": "tool_result",
            "tool_use_id": block["id"],  # CRITICAL: links THIS result to THIS specific request
            "content": json.dumps(result),
        })
    return tool_result_blocks


response = simulate_multi_tool_model_response()
print("=" * 70)
print("MODULE 2: REAL DISPATCH, THREE TOOLS REQUESTED IN ONE TURN")
print("=" * 70)

print("\nModel's single response requested THREE tool calls at once:")
for block in response["content"]:
    print(f"  id={block['id']} -> {block['name']}({block['input']})")

tool_results = dispatch_multi_tool_call(response)

print("\nDispatch layer's results, each correctly tagged with its tool_use_id:")
for tr in tool_results:
    tr_id = tr["tool_use_id"]
    tr_content = tr["content"]
    print(f"  tool_use_id={tr_id} -> {tr_content}")

# REAL verification: does every result correctly match back to its request?
request_ids = {b["id"] for b in response["content"]}
result_ids = {tr["tool_use_id"] for tr in tool_results}
print(f"\nRequest IDs:  {request_ids}")
print(f"Result IDs:   {result_ids}")
print(f"All requests correctly matched to a result: {request_ids == result_ids}")

print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: REAL DISPATCH, THREE TOOLS REQUESTED IN ONE TURN

Model's single response requested THREE tool calls at once:
  id=toolu_A -> check_sender_history({'sender_email': 'shobha.chopra@email.com'})
  id=toolu_B -> validate_fd_reference({'reference_number': 'BJ2019FD7717'})
  id=toolu_C -> lookup_product_catalog({'product_name': 'Smart Saver FD'})

Dispatch layer's results, each correctly tagged with its tool_use_id:
  tool_use_id=toolu_A -> {"sender_email": "shobha.chopra@email.com", "is_repeat_sender": true, "prior_email_count": 3, "has_unresolved_issue": true, "last_topic": "premature withdrawal penalty dispute"}
  tool_use_id=toolu_B -> {"reference_number": "BJ2019FD7717", "found": true, "customer_name": "Shobha Chopra", "status": "Closed (Premature)"}
  tool_use_id=toolu_C -> {"product_name": "Smart Saver FD", "exists": true, "interest_rate_percent": 7.65, "tenure_months": 18, "minimum_deposit_inr": 25000}

Request IDs:  {'toolu_A', 'toolu_C', 'toolu_B'}
Result IDs:   {'toolu_A

### Module 3: Measuring Tool-Selection Accuracy Across Multiple Tools

Build a labeled test set covering which of the THREE tools (or none) should be called for a given email, and measure real selection accuracy -- the genuinely harder problem this topic names.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Tool-selection accuracy across multiple tools, measured
# ------------------------------------------------------------------

# labeled test set: (email_text, correct_tools_to_call as a set)
LABELED_SELECTION_SET = [
    ("What is the penalty for my FD BJ2019FD7717?", {"validate_fd_reference"}),
    ("Tell me about the Smart Saver FD interest rate.", {"lookup_product_catalog"}),
    ("My personal loan EMI bounced, please help.", set()),  # neither tool applies
    ("Can you check my FD BJ2019FD7717 and also tell me about Senior Citizen FD rates?",
     {"validate_fd_reference", "lookup_product_catalog"}),  # BOTH needed
]


def simulate_tool_selection(email_text: str) -> set:
    """Simulates which tools a model would select, given the email
    text -- a simplified but REAL selection function testing the
    actual multi-tool discrimination problem this topic is about."""
    import re
    selected = set()
    if re.search(r'\b[A-Z]{2}\d{4}FD\d{4}\b', email_text):
        selected.add("validate_fd_reference")
    # naive product-name matching against the catalog
    text_lower = email_text.lower()
    for product_name in PRODUCT_CATALOG.keys():
        if product_name in text_lower:
            selected.add("lookup_product_catalog")
    return selected


print("=" * 70)
print("TOOL-SELECTION ACCURACY ACROSS MULTIPLE TOOLS (measured)")
print("=" * 70)

correct_count = 0
for email, correct_tools in LABELED_SELECTION_SET:
    selected_tools = simulate_tool_selection(email)
    is_correct = selected_tools == correct_tools
    correct_count += is_correct

    print(f"\nEmail: '{email[:60]}...'")
    correct_display = correct_tools if correct_tools else "(none)"
    selected_display = selected_tools if selected_tools else "(none)"
    result_label = "CORRECT" if is_correct else "*** WRONG ***"
    print(f"  Correct tools to call: {correct_display}")
    print(f"  Selected tools:        {selected_display}")
    print(f"  {result_label}")

accuracy = correct_count / len(LABELED_SELECTION_SET) * 100
print(f"\nTool-selection accuracy: {correct_count}/{len(LABELED_SELECTION_SET)} = {accuracy:.0f}%")

print("\nNotice the FOURTH case specifically tests whether BOTH tools get")
print("correctly identified together -- this is the genuinely HARDER")
print("multi-tool discrimination problem this topic names, distinct from")
print("simple single-tool yes/no triggering (Chapter 9 Topic 1).")

print("\nModule 3 complete. All Chapter 10 Topic 6 modules done.")
print()
print("=" * 70)
print("CHAPTER 10 TOPIC 6 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  Tool SELECTION (choosing correctly among several tools) is a
  genuinely harder problem than single-tool triggering -- demonstrated
  with a real labeled test set covering single-tool, no-tool, and
  BOTH-tools-needed cases.

  New tools (lookup_product_catalog, check_sender_history) should
  follow the SAME design template as validate_fd_reference: structured
  output, clear found/exists distinction, input echoed back.

  tool_use_id linkage becomes genuinely LOAD-BEARING, not just
  theoretical, once a single turn requests MULTIPLE different tools --
  demonstrated concretely with real dispatch handling three tools at once.

  Proactive tools (check_sender_history) vs reactive tools
  (validate_fd_reference, lookup_product_catalog) need different
  triggering strategies -- this is a real design decision, not a
  one-size-fits-all default.

  Tool composition (how multiple results combine into ONE final answer)
  is a new question multi-tool agents raise that single-tool agents
  never face -- important composition rules belong in the system
  prompt, not left to inference.
""")


TOOL-SELECTION ACCURACY ACROSS MULTIPLE TOOLS (measured)

Email: 'What is the penalty for my FD BJ2019FD7717?...'
  Correct tools to call: {'validate_fd_reference'}
  Selected tools:        {'validate_fd_reference'}
  CORRECT

Email: 'Tell me about the Smart Saver FD interest rate....'
  Correct tools to call: {'lookup_product_catalog'}
  Selected tools:        {'lookup_product_catalog'}
  CORRECT

Email: 'My personal loan EMI bounced, please help....'
  Correct tools to call: (none)
  Selected tools:        (none)
  CORRECT

Email: 'Can you check my FD BJ2019FD7717 and also tell me about Seni...'
  Correct tools to call: {'validate_fd_reference', 'lookup_product_catalog'}
  Selected tools:        {'validate_fd_reference', 'lookup_product_catalog'}
  CORRECT

Tool-selection accuracy: 4/4 = 100%

Notice the FOURTH case specifically tests whether BOTH tools get
correctly identified together -- this is the genuinely HARDER
multi-tool discrimination problem this topic names, distinct from
